# 😊 얼굴 랜드마크 검출 (Face Landmark Detection)

MediaPipe **FaceLandmarker**는 한 장의 이미지에서 **478개**의 얼굴 특징점(landmark)을 검출합니다.
눈·눈썹·입술·얼굴 윤곽까지 세밀하게 잡아내어 **블링크 감지, 표정 인식, AR 필터, 아바타 생성** 등에 활용됩니다.

| 항목 | 내용 |
|------|------|
| **태스크** | FaceLandmarker |
| **모델** | `face_landmarker.task` |
| **랜드마크 수** | 478 개 (이전 Solution: 468 + iris 10) |
| **출력** | 3D 좌표 (x, y, z) + visibility/presence |
| **지원 모드** | IMAGE, VIDEO, LIVE_STREAM |

```bash
pip install mediapipe opencv-python numpy matplotlib
```


---
## 1. FaceLandmarker 설정

In [ ]:
# 기본 임포트
import cv2
import numpy as np
import matplotlib.pyplot as plt
import mediapipe as mp

from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# 그리기 유틸/스타일 (이전 Solution API의 편의 도구)
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

print('MediaPipe:', mp.__version__)


In [ ]:
# 모델 경로 (노트북 파일 기준 상대경로)
MODEL_PATH = 'models/face_landmarker.task'

BaseOptions = python.BaseOptions
FaceLandmarker = vision.FaceLandmarker
FaceLandmarkerOptions = vision.FaceLandmarkerOptions
VisionRunningMode = vision.RunningMode

# 옵션 구성
options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=VisionRunningMode.IMAGE,   # 정적 이미지 모드
    num_faces=1,                            # 검출할 최대 얼굴 수
    min_face_detection_confidence=0.5,
    min_face_presence_confidence=0.5,
    min_tracking_confidence=0.5,
)

# 영상 모드용 옵션 (running_mode만 다름)
options_video = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=VisionRunningMode.VIDEO,
    num_faces=1,
)

# 스트림 모드용 옵션 (웹캠 실시간)
options_stream = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=VisionRunningMode.LIVE_STREAM,
    num_faces=1,
    result_callback=lambda result, output_image, timestamp_ms: None,
)

print('옵션 구성 완료')


**주요 옵션 설명**

| 옵션 | 기본 | 설명 |
|------|------|------|
| `running_mode` | IMAGE | `IMAGE` / `VIDEO` / `LIVE_STREAM` |
| `num_faces` | 1 | 동시 검출 얼굴 수 |
| `min_face_detection_confidence` | 0.5 | 얼굴 검출 신뢰도 임계값 |
| `min_face_presence_confidence` | 0.5 | 얼굴 존재 신뢰도 임계값 |
| `min_tracking_confidence` | 0.5 | 추적 신뢰도 임계값 (VIDEO/STREAM) |

> ⚠️ `running_mode`에 맞는 `detect()` / `detect_for_video()` / `detect_async()` 호출을 섞어 쓰면 에러가 납니다.


---
## 2. 478개 얼굴 랜드마크 개요

MediaPipe FaceLandmarker는 **478개**의 3D 랜드마크를 반환합니다.
이전 Solutions API의 468점에 **홍채(iris) 10점**이 추가된 형태입니다.

```
468 (기본 얼굴 메쉬) + 10 (양쪽 홍채, 각 5점) = 478
```

각 랜드마크는 다음 필드를 가집니다:

| 필드 | 범위 | 설명 |
|------|------|------|
| `x` | 0.0 ~ 1.0 | 이미지 폭 기준 정규화 좌표 |
| `y` | 0.0 ~ 1.0 | 이미지 높이 기준 정규화 좌표 |
| `z` | - | 깊이 (얼굴 중심이 0, 카메라 방향이 음수) |
| `visibility` | 0.0 ~ 1.0 | 외부에 보이는 정도 |
| `presence` | 0.0 ~ 1.0 | 장면에 존재할 확률 |

> 💡 `x, y`는 정규화되어 있으므로 실제 픽셀 좌표는 `x * width`, `y * height` 로 변환합니다.


---
## 3. 주요 영역별 랜드마크 인덱스

In [ ]:
# 자주 쓰는 영역별 대표 인덱스 (0-base, 478 기준)
FACE_REGIONS = {
    '오른쪽 눈':    list(range(33, 42 + 1)) + [145, 159, 160, 161, 163, 246],
    '왼쪽 눈':      list(range(463, 472 + 1)) + [374, 386, 387, 388, 390, 466],
    '오른쪽 눈썹':  list(range(46, 56 + 1)) + [285, 295, 300, 301, 332, 338, 339, 356],
    '왼쪽 눈썹':    list(range(276, 286 + 1)) + [70, 63, 105, 66, 107, 336],
    '오른쪽 홍채':  [468, 469, 470, 471, 472],
    '왼쪽 홍채':    [473, 474, 475, 476, 477],
    '코':          [1, 2, 5, 4, 6, 19, 20, 94, 125, 168, 197, 195, 45, 51, 48, 64, 98, 115, 131, 220, 234, 235],
    '입술 (외곽)':  list(range(61, 81 + 1)) + [291, 308, 324, 346, 375, 405, 409, 416],
    '입술 (내부)':  [78, 95, 88, 178, 87, 14, 317, 402, 318, 324, 308],
    '얼굴 윤곽':    [10, 338, 297, 332, 284, 251, 389, 356, 454, 323, 361, 288, 397, 365, 379, 378,
                   400, 377, 152, 148, 176, 149, 150, 136, 172, 58, 132, 93, 234, 127, 162, 21,
                   54, 103, 67, 109],
}

for region, idx in FACE_REGIONS.items():
    print(f'{region:12s}: {len(idx):3d}점  예) {idx[:6]}...')

print(f'\n전체 랜드마크: 478개')


**그림으로 보는 영역 배치** (대표 인덱스)

```
                눈썹(L)   눈썹(R)
                  46~       ~276
            눈(L)               눈(R)
          463~472               33~42
                  코
                  1,4,6
                입술
              61~81, 291~
        ┌────────얼굴 윤곽────────┐
       10                       454
        └────────────────────────┘
```

> 실제 인덱스 맵은 [MediaPipe Face Mesh](https://github.com/google-ai-edge/mediapipe/blob/master/mediapipe/python/solutions/face_mesh_connections.py) 문서를 참고하세요.


---
## 4. drawing_utils & drawing_styles 로 그리기

In [ ]:
# FaceLandmarker 결과(478점)를 이전 Solution API용 drawing_utils 로 그리기 위해서는
# NormalizedLandmarkList 형태로 변환해야 합니다.
from mediapipe.framework.formats import landmark_pb2

def draw_face_landmarks(image, face_landmarks_list):
    """FaceLandmarker 결과를 drawing_utils 로 시각화"""
    annotated = image.copy()
    for face_lm in face_landmarks_list:
        # protobuf NormalizedLandmarkList 로 변환
        proto_list = landmark_pb2.NormalizedLandmarkList()
        for lm in face_lm:
            p = proto_list.landmark.add()
            p.x = lm.x
            p.y = lm.y
            p.z = lm.z
            if hasattr(lm, 'visibility'):
                p.visibility = lm.visibility
            if hasattr(lm, 'presence'):
                p.presence = lm.presence

        # 랜드마크 점 + 메쉬(tesselation) + 윤곽(contour) 그리기
        mp_drawing.draw_landmarks(
            annotated,
            proto_list,
            mp.solutions.face_mesh.FACEMESH_TESSELATION,
            landmark_drawing_spec=None,
            connection_drawing_spec=mp_drawing_styles.get_default_face_mesh_tesselation_style(),
        )
        mp_drawing.draw_landmarks(
            annotated,
            proto_list,
            mp.solutions.face_mesh.FACEMESH_CONTOURS,
            landmark_drawing_spec=mp_drawing_styles.get_default_face_mesh_contours_style(),
            connection_drawing_spec=mp_drawing_styles.get_default_face_mesh_contours_style(),
        )
        mp_drawing.draw_landmarks(
            annotated,
            proto_list,
            mp.solutions.face_mesh.FACEMESH_IRISES,
            landmark_drawing_styles.get_default_face_mesh_iris_connections_style(),
            connection_drawing_styles.get_default_face_mesh_iris_connections_style(),
        )
    return annotated

print('draw_face_landmarks() 정의 완료')


**drawing_styles 가 제공하는 스타일**

| 함수 | 용도 |
|------|------|
| `get_default_face_mesh_tesselation_style()` | 얼굴 메쉬(삼각형망) 색상 |
| `get_default_face_mesh_contours_style()` | 눈/입술/윤곽 윤곽선 색상 |
| `get_default_face_mesh_iris_connections_style()` | 홍채 연결선 색상 |

**사용 가능한 연결(Contour) 집합** (`mp.solutions.face_mesh`):
- `FACEMESH_TESSELATION` — 전체 삼각형 메쉬
- `FACEMESH_CONTOURS` — 눈·눈썹·입·윤곽
- `FACEMESH_IRISES` — 홍채
- `FACEMESH_FACE_OVAL` — 얼굴 타원 윤곽만


---
## 5. 정적 이미지에서 얼굴 랜드마크 검출

In [ ]:
import urllib.request, os

# 샘플 얼굴 이미지 다운로드 (없을 때만)
SAMPLE_PATH = 'sample_face.jpg'
if not os.path.exists(SAMPLE_PATH):
    url = 'https://storage.googleapis.com/mediapipe-assets/portrait.jpg'
    urllib.request.urlretrieve(url, SAMPLE_PATH)
    print('다운로드 완료:', SAMPLE_PATH)
else:
    print('이미 존재:', SAMPLE_PATH)

# 이미지 읽기
image = cv2.imread(SAMPLE_PATH)
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
H, W = image_rgb.shape[:2]
print(f'이미지 크기: {W} x {H}')


In [ ]:
# IMAGE 모드로 검출
with FaceLandmarker.create_from_options(options) as detector:
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)
    result = detector.detect(mp_image)

print('검출된 얼굴 수:', len(result.face_landmarks))
if result.face_landmarks:
    print('첫 얼굴 랜드마크 수:', len(result.face_landmarks[0]))
    lm0 = result.face_landmarks[0][1]   # 코 끝
    print(f'코 끝(idx=1): x={lm0.x:.3f} y={lm0.y:.3f} z={lm0.z:.3f}')
    print(f'  → 픽셀 좌표: ({int(lm0.x*W)}, {int(lm0.y*H)})')


In [ ]:
# 시각화
annotated = draw_face_landmarks(image_rgb, result.face_landmarks)

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1); plt.imshow(image_rgb); plt.title('원본'); plt.axis('off')
plt.subplot(1, 2, 2); plt.imshow(annotated); plt.title('얼굴 랜드마크 478'); plt.axis('off')
plt.tight_layout(); plt.show()


In [ ]:
# 주요 영역만 강조해서 그리기 (눈/입술/윤곽)
def draw_regions(image, face_landmarks, region_indices, color=(255, 0, 0), radius=3):
    """특정 영역 인덱스만 점으로 그리기"""
    img = image.copy()
    H, W = img.shape[:2]
    for face_lm in face_landmarks:
        for idx in region_indices:
            if idx < len(face_lm):
                lm = face_lm[idx]
                cx, cy = int(lm.x * W), int(lm.y * H)
                cv2.circle(img, (cx, cy), radius, color, -1)
    return img

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
regions_to_show = [
    ('오른쪽 눈', (255, 0, 0)),
    ('왼쪽 눈', (0, 0, 255)),
    ('입술 (외곽)', (255, 0, 255)),
    ('오른쪽 홍채', (255, 255, 0)),
    ('얼굴 윤곽', (0, 255, 0)),
    ('코', (0, 255, 255)),
]
for ax, (name, color) in zip(axes.flat, regions_to_show):
    ax.imshow(draw_regions(image_rgb, result.face_landmarks, FACE_REGIONS[name], color=color))
    ax.set_title(name); ax.axis('off')
plt.tight_layout(); plt.show()


---
## 6. 웹캠 실시간 얼굴 랜드마크

In [ ]:
# LIVE_STREAM 모드 + 비동기 콜백 방식
import time

latest_result = {'data': None}

def on_result(result, output_image, timestamp_ms):
    """프레임마다 호출되는 콜백 — 결과를 전역에 저장"""
    latest_result['data'] = result

stream_options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=VisionRunningMode.LIVE_STREAM,
    num_faces=1,
    result_callback=on_result,
)

# 웹캡 루프 (이 셀을 실행하면 'q' 키로 종료)
cap = cv2.VideoCapture(0)
print('웹캠 시작 — 종료하려면 창에서 q 키')

with FaceLandmarker.create_from_options(stream_options) as detector:
    while cap.isOpened():
        ok, frame = cap.read()
        if not ok:
            break
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
        timestamp_ms = int(time.time() * 1000)
        detector.detect_async(mp_image, timestamp_ms)

        # 최신 결과로 화면 갱신
        res = latest_result['data']
        if res and res.face_landmarks:
            annotated = draw_face_landmarks(frame_rgb, res.face_landmarks)
        else:
            annotated = frame_rgb

        cv2.imshow('Face Landmark (q to quit)', cv2.cvtColor(annotated, cv2.COLOR_RGB2BGR))
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()


> 💡 **VIDEO 모드**를 쓰면 동기식으로 처리할 수도 있습니다:
> ```python
> result = detector.detect_for_video(mp_image, timestamp_ms)
> ```
> 처리 속도가 느리면 프레임이 버퍼에 쌓이므로 실시간성이 중요할 때는 `LIVE_STREAM`이 더 부드럽습니다.


---
## 7. 실전: 눈 깜빡임(Eye Aspect Ratio, EAR) 감지

**Eye Aspect Ratio(EAR)** 는 눈의 세로/가로 비율로 눈이 감겼는지 판단하는 고전적 지표입니다.

```
        P1
   P6 ╱───╲ P2     눈꼬리 6개 점으로 계산
    │   ●   │
   P5 ╲───╱ P3
        P4

EAR = (||P2-P6|| + ||P3-P5||) / (2 * ||P1-P4||)
```

- **눈 뜸**: EAR ≈ 0.25 ~ 0.35
- **눈 감음**: EAR < 0.20 (임계값)
- 한 프레임이 임계값 아래로 떨어졌다가 다시 올라오면 **1회 깜빡임** 으로 카운트


In [ ]:
# MediaPipe 눈 랜드마크 인덱스 (EAR 계산용 6점)
# 오른쪽 눈: 33(왼쪽끝), 133(오른쪽끝), 159(위), 145(아래), 158(위위), 153(아래아래)
# 왼쪽 눈:   362, 263, 386, 374, 385, 380
RIGHT_EYE = [33, 133, 160, 144, 158, 153]
LEFT_EYE  = [362, 263, 385, 373, 387, 380]
# 일반적으로 쓰는 6점 순서: [outer, inner, top, bottom, top2, bottom2]

def ear_distance(p1, p2):
    """두 랜드마크 사이의 유클리드 거리"""
    return np.linalg.norm(np.array([p1.x, p1.y]) - np.array([p2.x, p2.y]))

def eye_aspect_ratio(landmarks, eye_indices):
    """EAR 계산"""
    P1, P2, P3, P4, P5, P6 = [landmarks[i] for i in eye_indices]
    # 세로 거리 2개
    d1 = ear_distance(P2, P6)
    d2 = ear_distance(P3, P5)
    # 가로 거리 1개
    d3 = ear_distance(P1, P4)
    ear = (d1 + d2) / (2.0 * d3)
    return ear

# 단일 이미지에서 EAR 측정해보기
if result.face_landmarks:
    face_lm = result.face_landmarks[0]
    ear_r = eye_aspect_ratio(face_lm, RIGHT_EYE)
    ear_l = eye_aspect_ratio(face_lm, LEFT_EYE)
    ear_avg = (ear_r + ear_l) / 2.0
    print(f'오른쪽 눈 EAR: {ear_r:.3f}')
    print(f'왼쪽 눈  EAR: {ear_l:.3f}')
    print(f'평균 EAR    : {ear_avg:.3f}')
    print('→ 눈 상태:', '감김' if ear_avg < 0.20 else '뜸')


In [ ]:
# 실시간 블링크 카운터 (웹캠)
EAR_THRESHOLD = 0.21   # 임계값 (사용자에 따라 조정)
EAR_CONSEC_FRAMES = 2  # 연속 프레임 수 (노이즈 방지)

blink_count = 0
frame_counter = 0
ear_history = []

stream_options2 = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=VisionRunningMode.LIVE_STREAM,
    num_faces=1,
    result_callback=on_result,   # 위에서 정의한 콜백 재사용
)

cap = cv2.VideoCapture(0)
print('블링크 감지 시작 — 종료하려면 q 키')

with FaceLandmarker.create_from_options(stream_options2) as detector:
    while cap.isOpened():
        ok, frame = cap.read()
        if not ok:
            break
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
        detector.detect_async(mp_image, int(time.time() * 1000))

        res = latest_result['data']
        display = frame.copy()

        if res and res.face_landmarks:
            face_lm = res.face_landmarks[0]
            ear = (eye_aspect_ratio(face_lm, RIGHT_EYE) +
                   eye_aspect_ratio(face_lm, LEFT_EYE)) / 2.0
            ear_history.append(ear)

            if ear < EAR_THRESHOLD:
                frame_counter += 1
            else:
                if frame_counter >= EAR_CONSEC_FRAMES:
                    blink_count += 1
                frame_counter = 0

            # 화면에 EAR / 카운트 표시
            cv2.putText(display, f'EAR: {ear:.3f}', (20, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
            color = (0, 0, 255) if ear < EAR_THRESHOLD else (0, 255, 0)
            cv2.putText(display, f'BLINKS: {blink_count}', (20, 70),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

        cv2.imshow('Blink Detector (q to quit)', display)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()
print(f'\n총 깜빡임 횟수: {blink_count}')


In [ ]:
# EAR 시계열 그래프
if ear_history:
    plt.figure(figsize=(10, 4))
    plt.plot(ear_history, label='EAR')
    plt.axhline(EAR_THRESHOLD, color='r', linestyle='--', label=f'Threshold={EAR_THRESHOLD}')
    plt.title('Eye Aspect Ratio over frames')
    plt.xlabel('frame'); plt.ylabel('EAR'); plt.legend(); plt.grid(True)
    plt.tight_layout(); plt.show()


---
## 정리

| 항목 | 내용 |
|------|------|
| **태스크** | `vision.FaceLandmarker` |
| **모델** | `models/face_landmarker.task` |
| **랜드마크 수** | 478 (468 + 홍채 10) |
| **좌표 형식** | 정규화 (x, y) + 깊이 z + visibility/presence |
| **실행 모드** | IMAGE / VIDEO / LIVE_STREAM |
| **주요 영역** | 눈·눈썹·코·입술·홍채·얼굴 윤곽 |
| **그리기** | `drawing_utils` + `drawing_styles`, FACEMESH_TESSELATION/CONTOURS/IRISES |
| **실전 예제** | EAR(눈 가로/세로 비율) 기반 블링크 감지 |

**핵심 포인트**
- 478점은 **정규화 좌표**이므로 픽셀 좌표로 변환하려면 `× width`, `× height`
- `drawing_utils`는 구 Solution API 용이므로 **NormalizedLandmarkList 변환**이 필요
- LIVE_STREAM 모드는 **콜백**으로 결과를 받고, 호출 스레드가 아닌 곳에서 화면 갱신
- EAR 임계값은 **사용자·조명·카메라**에 따라 보정 필요 (기본 0.20~0.22)


---
## 연습 문제

**연습 1.** 얼굴 윤곽(`FACE_REGIONS['얼굴 윤곽']`) 점들만 이어 **얼굴 타원 마스크**를 만들어 배경을 블러(blur) 처리하는 함수를 작성하세요.

**연습 2.** 두 눈의 홍초 인덱스(468~477)를 이용해 **시선 방향**을 추정해 보세요.
- 힌트: 홍채 중심(468/473)과 눈 중심(33~42 평균 / 362~372 평균)의 상대 위치로 좌/우/상/하 판단

**연습 3.** 입술 외곽 랜드마크 사이의 거리를 이용해 **입 벌림 정도(Mouth Aspect Ratio)** 를 계산하고, 일정 임계값 이상이면 "말하는 중" 으로 표시하는 실시간 디스플레이를 만드세요.

**연습 4.** `num_faces=2`로 설정한 뒤 두 사람 얼굴을 동시에 검출하고, 각 얼굴의 EAR를 별도로 계산하여 화면에 `A: n회 / B: m회` 형식으로 깜빡임 횟수를 표시하세요.
